# Missing Value Handling – Loan Prediction Dataset

## Objective

The goal of this notebook is to handle missing values in the dataset using different techniques and select the most appropriate strategy while preserving data quality.

Techniques explored:
- Dropping rows
- Mean imputation
- Median imputation
- Mode imputation
- Data loss comparison

KAGGLE SETUP

In [58]:
!pip install kaggle --quiet

In [6]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ksaivardhangoud","key":"8eb1bfbeaa469b52cbd57d27899c82e5"}'}

In [7]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d hossamhibrahem/loan-prediction-analytics-vidhya
!unzip loan-prediction-analytics-vidhya.zip

Dataset URL: https://www.kaggle.com/datasets/hossamhibrahem/loan-prediction-analytics-vidhya
License(s): unknown
  0% 0.00/13.6k [00:00<?, ?B/s]
100% 13.6k/13.6k [00:00<00:00, 39.1MB/s]
Archive:  loan-prediction-analytics-vidhya.zip
  inflating: sample_submission_49d68Cx.csv  
  inflating: test_lAUu6dG.csv        
  inflating: train_ctrUa4K.csv       


In [8]:
!ls

kaggle.json			      sample_submission_49d68Cx.csv
loan-prediction-analytics-vidhya.zip  test_lAUu6dG.csv
sample_data			      train_ctrUa4K.csv


PROCESS STARTS

In [9]:
import pandas as pd


## Load Dataset

In [59]:
df = pd.read_csv("train_ctrUa4K.csv")
df.shape

(614, 13)

The dataset contains 614 rows and 13 columns.

## Missing Value Analysis

In [60]:
df.isnull().sum()

,0
Loan_ID,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14


Missing values are present in:
- LoanAmount
- Loan_Amount_Term
- Credit_History
- Some categorical columns

## Method 1: Dropping Rows with Missing Values

In [61]:
df_drop = df.dropna()

print("Original:", df.shape)
print("After Drop:", df_drop.shape)

loss_percentage = (df.shape[0] - df_drop.shape[0]) / df.shape[0] * 100
print("Percentage of Data Lost:", loss_percentage)

Original: (614, 13)
After Drop: (480, 13)
Percentage of Data Lost: 21.824104234527688


Dropping rows resulted in approximately 22% data loss.  
Since the dataset is relatively small, this approach is not ideal.

## Identify Numerical Columns

In [62]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
df[num_cols].isnull().sum()

,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14
Credit_History,50


## Skewness Analysis

In [63]:
df[['LoanAmount','Loan_Amount_Term','Credit_History']].skew()

,0
LoanAmount,2.677552
Loan_Amount_Term,-2.362414
Credit_History,-1.882361


- LoanAmount is highly right-skewed.
- Loan_Amount_Term is left-skewed.
- Credit_History behaves like a binary categorical variable.

##Identify categorical columns

In [66]:
cat_cols = df_clean.select_dtypes(include=['object']).columns
df[cat_cols].isnull().sum()

,0
Loan_ID,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
Property_Area,0
Loan_Status,0


## Apply Imputation Strategy

In [68]:
df_clean = df.copy()

# Numerical (skewed → median)
df_clean['LoanAmount'] = df_clean['LoanAmount'].fillna(df_clean['LoanAmount'].median())
df_clean['Loan_Amount_Term'] = df_clean['Loan_Amount_Term'].fillna(df_clean['Loan_Amount_Term'].median())

# Binary-like → mode
df_clean['Credit_History'] = df_clean['Credit_History'].fillna(df_clean['Credit_History'].mode()[0])

df_clean['Gender'] = df_clean['Gender'].fillna(df_clean['Gender'].mode()[0])

df_clean['Married'] = df_clean['Married'].fillna(df_clean['Married'].mode()[0])

df_clean['Dependents'] = df_clean['Dependents'].fillna(df_clean['Dependents'].mode()[0])

df_clean['Self_Employed'] = df_clean['Self_Employed'].fillna(df_clean['Self_Employed'].mode()[0])

## Verification – Check Remaining Missing Values

In [69]:
df_clean.isnull().sum()

,0
Loan_ID,0
Gender,0
Married,0
Dependents,0
Education,0
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,0
Loan_Amount_Term,0


All missing values have been successfully handled.

## Save Cleaned Dataset

In [70]:
df_clean.to_csv("cleaned_loan_dataset.csv", index=False)

# Final Missing Value Handling Summary

1. Dropping rows removed ~22% of the dataset and was rejected.
2. LoanAmount imputed using median due to high right skew.
3. Loan_Amount_Term imputed using median due to skewness.
4. Credit_History treated as categorical and imputed using mode.
5. Categorical columns filled using mode.
6. Final dataset contains 614 rows with no missing values.
7. The Final dataset contains zero missing values means ready for model building.

The dataset is now ready for encoding and model building.